# Chapter 2.5: Data coverage and variant representation across samples
## Create AAV6-ML figures for Chapter 3 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 22/04/2026
Finished: ...

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'2.5_n_sample'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

In [ ]:
# comare and corr two tables
def compare_two_tables(
    df1,
    df2,
    name1="sample_1",
    name2="sample_2",
    compare_cols=None,
    key_col="AA_sequence",
    log_scale_cols=None,
    add_diagonal=True,
    alpha=0.25,
    point_size=8,
    figsize_per_panel=(5.5, 5),
    save=False,
    output_path=None,
    file_name=None,
    show=True
):
    """
    Compare two tables across one or multiple numeric columns by merging on key_col.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Tables to compare
    name1, name2 : str
        Names shown in axis labels
    compare_cols : list of str
        Numeric columns to compare, e.g. ["Log2_enrichment", "RPM"]
    key_col : str
        Merge key, default = "AA_sequence"
    log_scale_cols : list of str
        Columns that should be plotted on log-log axes, e.g. ["RPM"]
    add_diagonal : bool
        Whether to add y=x diagonal
    alpha : float
        Point transparency
    point_size : int
        Scatter point size
    figsize_per_panel : tuple
        Width, height per subplot
    save : bool
        Whether to save figure
    output_path : str or Path
        Save directory
    file_name : str
        Output filename
    show : bool
        Whether to show plot

    Returns
    -------
    merged : pd.DataFrame
        Merged comparison table
    corr_df : pd.DataFrame
        Correlation summary table
    """

    if compare_cols is None:
        compare_cols = ["Log2_enrichment", "RPM"]

    if log_scale_cols is None:
        log_scale_cols = []

    # keep only needed columns
    cols1 = [key_col] + compare_cols
    cols2 = [key_col] + compare_cols

    a = df1[cols1].copy()
    b = df2[cols2].copy()

    # rename value columns
    a = a.rename(columns={col: f"{col}_{name1}" for col in compare_cols})
    b = b.rename(columns={col: f"{col}_{name2}" for col in compare_cols})

    # merge
    merged = a.merge(b, on=key_col, how="inner")

    # correlation results
    results = []

    # figure layout
    n = len(compare_cols)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows)
    )

    # make axes iterable
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).reshape(-1)

    for i, col in enumerate(compare_cols):
        ax = axes[i]

        xcol = f"{col}_{name1}"
        ycol = f"{col}_{name2}"

        sub = merged[[xcol, ycol]].dropna().copy()

        # correlations
        if len(sub) > 1:
            pearson_r, pearson_p = pearsonr(sub[xcol], sub[ycol])
            spearman_r, spearman_p = spearmanr(sub[xcol], sub[ycol])
        else:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_r, spearman_p = np.nan, np.nan

        results.append({
            "column": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "n_points": len(sub)
        })

        # scatter
        sns.scatterplot(
            data=sub,
            x=xcol,
            y=ycol,
            s=point_size,
            alpha=alpha,
            ax=ax
        )

        # log scaling if requested
        if col in log_scale_cols:
            sub_pos = sub[(sub[xcol] > 0) & (sub[ycol] > 0)].copy()
            ax.clear()
            sns.scatterplot(
                data=sub_pos,
                x=xcol,
                y=ycol,
                s=point_size,
                alpha=alpha,
                ax=ax
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            plot_data = sub_pos
        else:
            plot_data = sub

        # diagonal
        if add_diagonal and len(plot_data) > 0:
            lims = [
                min(plot_data[xcol].min(), plot_data[ycol].min()),
                max(plot_data[xcol].max(), plot_data[ycol].max())
            ]
            ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.6)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        ax.set_xlabel(f"{col} ({name1})")
        ax.set_ylabel(f"{col} ({name2})")
        ax.set_title(col)

        ax.text(
            0.05, 0.95,
            f"Pearson r = {pearson_r:.4f}\nSpearman ρ = {spearman_r:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        sns.despine(ax=ax)

    # remove unused axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        import os
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            col_string = "_".join(compare_cols)
            file_name = f"compare_{name1}_vs_{name2}_{col_string}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    corr_df = pd.DataFrame(results)

    return merged, corr_df

## 3. Script Functions

### 3.1. variant presence distribution across variant presence

In [ ]:
def variant_presence_distribution(
    table,
    tissue,
    extraction,
    col="n_samples_present",
    n_sample_order=[0, 1, 2, 3, 4, 5, 6],
    log_scale = False,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot the distribution of pooled variants across the number of biological
    samples in which they are detected.

    The function filters a pooled table for one tissue and extraction type,
    counts how many unique AA variants fall into each `n_samples_present`
    category, converts these counts into percentages, and visualizes the result
    as a bar plot. Absolute variant counts are shown above each bar.

    Parameters
    ----------
    table : pd.DataFrame
        Pooled input table. Expected to contain at least:
        ['AA_sequence', 'Tissue', 'Extraction_type', col, 'Log2_enrichment'].
    tissue : str
        Tissue to analyze, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to analyze, e.g. 'gDNA' or 'cDNA'.
    col : str, default='n_samples_present'
        Column defining in how many biological samples a variant is detected.
    n_sample_order : list, optional
        Explicit order of `n_samples_present` groups to include on the x-axis.
        If None, this is derived automatically from `include_zero`.
    include_zero : bool, default=False
        Whether to include variants detected in 0 biological samples.
    title : bool, default=True
        Whether to display a plot title.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Output directory for saving the figure. Required if save=True.
    file_name : str, optional
        Output file name. If None, a default file name is generated.

    Returns
    -------
    summary : pd.DataFrame
        Summary table containing:
        - `n_samples_present`
        - `n_variants`
        - `mean_log2_enrichment`
        - `percent_variants`

    Notes
    -----
    - The bar height reflects the percentage of measured variants in each
      `n_samples_present` class.
    - The absolute number of unique AA variants is annotated above each bar.
    - Missing `n_samples_present` classes are filled with zero so that all
      requested x-axis groups are shown.
    """

    # ----------------------------------------------------------
    # Filter pooled table for selected tissue and extraction type
    # ----------------------------------------------------------
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ].copy()

    # ----------------------------------------------------------
    # Summarize number of variants per presence class
    # ----------------------------------------------------------
    summary = (
        df.groupby(col, as_index=False)
        .agg(
            n_variants=("AA_sequence", "nunique"),
            mean_log2_enrichment=("Log2_enrichment", "mean")
        )
    )

    # ----------------------------------------------------------
    # Reindex to ensure all requested presence classes are shown
    # ----------------------------------------------------------
    summary = (
        summary.set_index(col)
        .reindex(n_sample_order)
        .fillna(0)
        .reset_index()
    )

    # ----------------------------------------------------------
    # Convert absolute counts into percentages
    # ----------------------------------------------------------
    total_variants = summary["n_variants"].sum()
    summary["percent_variants"] = 100 * summary["n_variants"] / total_variants

    # ----------------------------------------------------------
    # Set style
    # ----------------------------------------------------------
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # ----------------------------------------------------------
    # Plot bar chart
    # ----------------------------------------------------------
    bars = ax.bar(
        summary[col],
        summary["percent_variants"],
        color="#4C78A8",
        edgecolor="black",
        linewidth=1
    )

    # ----------------------------------------------------------
    # Add absolute counts above bars
    # ----------------------------------------------------------
    ymax = summary["percent_variants"].max()
    offset = ymax * 0.02 if ymax > 0 else 0.5

    for bar, n in zip(bars, summary["n_variants"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{int(n):,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    # ----------------------------------------------------------
    # Labels and optional title
    # ----------------------------------------------------------
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel("Percent of measured variants (%)")
    ax.set_xticks(n_sample_order)

    if title:
        ax.set_title(f"Variants detected across {extraction} {tissue} samples")

    # ----------------------------------------------------------
    # Background and axis settings
    # ----------------------------------------------------------
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

    # expand ylim so count labels fit comfortably
    if log_scale:
        ax.set_yscale ('log', base=10)

    plt.tight_layout()

    # ----------------------------------------------------------
    # Save figure
    # ----------------------------------------------------------
    if save:
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"variant_presence_distribution_{extraction}_{tissue}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()
    return summary

### 3.2. Boxplot of mean count/RPM/log2_enrichment distribution across variant presence

In [ ]:
def Mean_RPM_count_distribution_by_sample_presence(
    table,
    tissue,
    extraction,
    value='RPM',
    n_sample_order=[0, 1, 2, 3, 4, 5, 6],
    mean_line=False,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):

    """
    Plot the distribution of pooled per-variant values grouped by how many
    biological samples a variant is detected in.

    The function expects a pooled table in which each row represents one
    AA sequence for a given tissue and extraction type, together with the
    corresponding `n_samples_present` annotation. It visualizes the selected
    variable (`value`) across user-defined presence classes using a boxplot.

    Optionally, a red line indicating the mean value per
    `n_samples_present` group can be overlaid.

    Parameters
    ----------
    table : pd.DataFrame
        Pooled table containing one row per variant and condition. Expected to
        include at least the columns:
        ['AA_sequence', 'Tissue', 'Extraction_type', 'n_samples_present',
         value, 'Log2_enrichment'].
    tissue : str
        Tissue to analyze, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to analyze, e.g. 'gDNA' or 'cDNA'.
    value : str, default='RPM'
        Column to visualize on the y-axis, e.g. 'RPM', 'Count',
        or 'Log2_enrichment'
    n_sample_order : list, default=[0, 1, 2, 3, 4, 5, 6]
        List defining which `n_samples_present` groups should be included
        in the plot and in which order they should appear on the x-axis.
    mean_line : bool, default=False
        Whether to overlay the mean value per `n_samples_present` group as a
        red line with markers.
    title : bool, default=True
        Whether to display a plot title.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Output directory for saving the figure. Required if save=True.
    file_name : str, optional
        Output file name. If None, a default file name is generated.

    Returns
    -------
    summary : pd.DataFrame
        Summary table containing the mean `Log2_enrichment` and mean `value`
        for each included `n_samples_present` group.

    Notes
    -----
    - This function operates on pooled / per-variant summary values
    - `n_samples_present` is assumed to already be present in the input table.
    - Only rows with `n_samples_present` values contained in `n_sample_order`
      are included.
    - The y-axis is displayed on a log scale for 'RPM' and 'Count', and on a
      linear scale for values such as 'Log2_enrichment'.
    - The boxplot shows the distribution of pooled per-variant values within
      each selected `n_samples_present` class.
    - The optional red line reflects the arithmetic mean of `value` within
      each selected `n_samples_present` class.
    """
    
    # Definition of n_sample column
    col="n_samples_present"

    #filter table for tissue and extraction combination
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction) &
        (table[col].isin(n_sample_order))
    ].copy()

    summary = (df.groupby([col], as_index = False)
        .agg({
            "Log2_enrichment": "mean",
            value: "mean"
            
        })
              )
    
    # set style of figure
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # plot figure
    sns.boxplot(
        data=df,
        x=col,
        y=value,
        order=n_sample_order,
        color="#4C78A8",
        showfliers=False,
        ax=ax
    )
     # add mean line
    if mean_line:
        ax.plot(
            summary[col],
            summary[value],
            color="red",
            marker="o",
            linewidth=1.8,
            markersize=5,
            label=f"Mean {value}"
        )

    # label of axis
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel(f"Mean {value}")


    # title
    if title:
        ax.set_title(f"{extraction} {tissue}: Mean {value} by sample presence")


    # background and axis settings
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    if value in ['Count', 'RPM']:
        ax.set_yscale("log", base=10)
    sns.despine(ax=ax)
    
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"Mean_{value}_by_sample_presence_{extraction}_{tissue}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=400, bbox_inches="tight")

    plt.show()
    return summary

### 3.3. Boxplot of count/RPM/log2_enrichment single values distribution across variant presence

In [ ]:
def RPM_count_distribution_by_sample_presence(
    table,
    tissue,
    extraction,
    value = 'RPM',
    n_sample_order = [0,1,2,3,4,5,6],
    no_pseudo = False,
    mean_line = False,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):

    """
    Plot the distribution of per-sample variant values grouped by how many
    biological samples a variant is detected in.
    
    The function combines sample-level observations from a long-format table with
    the corresponding `n_samples_present` annotation from `df_pooled`. This allows
    each individual observation to be assigned to a presence class
    (e.g. 0, 1, 2, ..., 6 detected biological samples) and visualized as a boxplot.
    
    Optionally, pseudo-supported observations can be filtered out for all variants
    with `n_samples_present >= 1`, while variants with `n_samples_present == 0`
    are retained. A mean trend line across presence classes can also be added.
    
    Parameters
    ----------
    table : pd.DataFrame
        Long-format table containing per-sample measurements. Expected to include
        at least the columns:
        ['AA_sequence', 'Tissue', 'Extraction_type', 'Replicate', value, 'Pseudo'].
    tissue : str
        Tissue to analyze, e.g. 'liver' or 'heart'.
    extraction : str
        Extraction type to analyze, e.g. 'gDNA' or 'cDNA'.
    value : str, default='RPM'
        Column to visualize on the y-axis, e.g. 'RPM', 'Count', or
        'Log2_enrichment'.
    n_sample_order : list, default=[0,1,2,3,4,5,6]
        Defines which `n_samples_present` groups should be plotted and in which
        order.
    no_pseudo : bool, default=False
        If True, apply the following filtering rule after merging:
        - keep all rows with `n_samples_present == 0`
        - for rows with `n_samples_present >= 1`, keep only observations with
          `Pseudo == 0`
    mean_line : bool, default=False
        Whether to overlay the mean value per `n_samples_present` group as a
        red line with markers.
    title : bool, default=True
        Whether to display a plot title.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Output directory for saving the figure. Required if save=True.
    file_name : str, optional
        Output file name. If None, a default file name is generated.
    
    Returns
    -------
    summary : pd.DataFrame
        Summary table containing the mean of `value` for each
        `n_samples_present` group after all applied filtering steps.
    
    Notes
    -----
    - `n_samples_present` is taken from the global `df_pooled` table and is not
      recalculated within this function.
    - Only biological replicates are included.
    - The boxplot shows the actual per-sample values, not per-variant pooled means.
    - For `value` equal to 'RPM' or 'Count', the y-axis is displayed on a log scale.
    - The optional red line reflects the arithmetic mean within each
      `n_samples_present` class.
    """
    # ----------------------------------------------------------
    # Definition of n_sample column
    # ----------------------------------------------------------
    col="n_samples_present"

    # ----------------------------------------------------------
    # get n_sample_present information from df pooled
    # ----------------------------------------------------------
    df_info = df_pooled[
        (df_pooled["Tissue"] == tissue) &
        (df_pooled["Extraction_type"] == extraction) &
        (df_pooled[col].isin(n_sample_order))
    ][['AA_sequence', col]].copy()
    
    # ----------------------------------------------------------
    #filter table for tissue and extraction combination
    # ----------------------------------------------------------
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction) &
        (table["Replicate"] == 'biological')
    ][['AA_sequence', value, "Pseudo"]].copy()

    # ----------------------------------------------------------
    # merge tables to get n_sample_present information for each variant
    # ----------------------------------------------------------
    df_merge = df.merge(df_info, on = 'AA_sequence', how = 'inner')

    # ----------------------------------------------------------
    # Optional filter for Pseudo if n_sample_present >= 1
    # ----------------------------------------------------------
    if no_pseudo:
        df_merge = df_merge[
            (df_merge[col] == 0) |
            ((df_merge[col] >= 1) & (df_merge["Pseudo"] == 0))
            ].copy()
    # ----------------------------------------------------------
    # Summary for optional mean line
    # ----------------------------------------------------------
    summary = (df_merge.groupby([col], as_index = False)
        .agg({
            value: "mean"
        })
              )
    # ----------------------------------------------------------
    # set style of figure
    # ----------------------------------------------------------
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # ----------------------------------------------------------
    # plot boxplot
    # ----------------------------------------------------------
    sns.boxplot(
        data=df_merge,
        x=col,
        y=value,
        order=n_sample_order,
        color="#4C78A8",
        showfliers=False,
        ax=ax
    )
    # ----------------------------------------------------------
    # add optional mean line
    # ----------------------------------------------------------
    if mean_line:
        ax.plot(
            summary[col],
            summary[value],
            color="red",
            marker="o",
            linewidth=1.8,
            markersize=5,
            label=f"Mean {value}"
        )

    # ----------------------------------------------------------
    # label of axis
    # ----------------------------------------------------------
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel(f"{value}")

    # ----------------------------------------------------------
    # Optional title
    # ----------------------------------------------------------
    if title:
        ax.set_title(f"{extraction} {tissue}: {value} by sample presence")

    # ----------------------------------------------------------
    # background and axis settings
    # ----------------------------------------------------------
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    
    if value in ['RPM', 'Count']:
        ax.set_yscale("log", base=10)
        
    sns.despine(ax=ax)
    
    plt.tight_layout()

    # ----------------------------------------------------------
    # Save figure
    # ----------------------------------------------------------

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"{value}_by_sample_presence_{extraction}_{tissue}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=400, bbox_inches="tight")

    plt.show()
    return summary

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. variant presence distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    variant_presence_distribution(
        table=df_pooled,
        tissue=tis,
        extraction=ext,
        log_scale = False,
        n_sample_order=[1, 2, 3, 4, 5, 6], 
        title=True,
        save=True,
        output_path=figures_dir/'2_variant_present_in_n_samples'/"no_zero", file_name=f"variant_presence_distribution_{ext}_{tis}_no0.png"
    )

### 5.2. Boxplot for log2_enrichment distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    Mean_RPM_count_distribution_by_sample_presence(table=df_pooled, 
                                                   tissue=tis, 
                                                   extraction=ext, 
                                                   n_sample_order=[1,2,3,4,5,6],
                                                   value='Log2_enrichment', 
                                                   mean_line=False, 
                                                   save = False
                                                  )

### 5.3. Boxplot of mean count/RPM distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    Mean_RPM_count_distribution_by_sample_presence(table=df_pooled, 
                                                   tissue=tis, 
                                                   extraction=ext, 
                                                   value='RPM', 
                                                   mean_line=True, 
                                                   save = False, 
                                                   output_path=figures_dir/'1_Count_RPM_boxplot'/'3_Mean_RPM_across_n_samples'
                                                  )

### 5.4. Boxplot of counts/RPM single values distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    RPM_count_distribution_by_sample_presence(
        table=df_long, 
        tissue=tis, 
        extraction=ext, 
        no_pseudo=False, 
        value='RPM', 
        mean_line=True, 
        save = False, 
        output_path=figures_dir/'1_Count_RPM_boxplot'/'4_RPM_across_n_samples'
    )

### 5.5. Boxplot of counts/RPM single values distribution across variant presence without pseudo measurments if n_sample_present >= 1

In [ ]:
for tis, ext in product (TISSUE, EXT):
    RPM_count_distribution_by_sample_presence(
        table=df_long, 
        tissue=tis, 
        extraction=ext, 
        value='RPM', 
        no_pseudo=True, 
        mean_line=True, 
        save = False, 
        output_path=figures_dir/'1_Count_RPM_boxplot'/'7_RPM_across_n_samples_no_pseudo'
    )